# 🪪 FajrGuard — FaceNet TFLite Export

Downloads vggface2 weights from GitHub Releases, exports to TFLite.
Zero manual steps. No `facenet_pytorch` import — avoids ALL dependency issues.

**Output:** `facenet_mobile.tflite`

In [ ]:
# ═══ Download weights from GitHub Releases ═════════════════════════
import os, torch, torch.nn as nn, torch.nn.functional as F, numpy as np
import urllib.request

WEIGHTS_PATH = "facenet_vggface2.pth"
WEIGHTS_URL  = "https://github.com/timesler/facenet-pytorch/releases/download/v2.2.9/20180402-114759-vggface2.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists(WEIGHTS_PATH) or os.path.getsize(WEIGHTS_PATH) < 50_000_000:
    print("Downloading vggface2 weights (~120 MB)...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print(f"Weights: {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB, Device: {DEVICE}")

In [ ]:
# ═══ Exact architecture from facenet-pytorch v2.6.0 ════════════════
# Key fix: Mixed_7a has 4 branches (branch3 = MaxPool2d), total=1792

class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_planes, eps=0.001, momentum=0.1, affine=True)
        self.relu = nn.ReLU(inplace=False)
    def forward(self, x):
        x = self.conv(x); x = self.bn(x); x = self.relu(x)
        return x

class Block35(nn.Module):
    def __init__(self, scale=1.0):
        super().__init__()
        self.scale = scale
        self.branch0 = BasicConv2d(256, 32, 1, 1)
        self.branch1 = nn.Sequential(BasicConv2d(256, 32, 1, 1), BasicConv2d(32, 32, 3, 1, 1))
        self.branch2 = nn.Sequential(BasicConv2d(256, 32, 1, 1), BasicConv2d(32, 32, 3, 1, 1), BasicConv2d(32, 32, 3, 1, 1))
        self.conv2d = nn.Conv2d(96, 256, 1, 1)
        self.relu = nn.ReLU(inplace=False)
    def forward(self, x):
        o = self.conv2d(torch.cat((self.branch0(x), self.branch1(x), self.branch2(x)), 1))
        return self.relu(o * self.scale + x)

class Block17(nn.Module):
    def __init__(self, scale=1.0):
        super().__init__()
        self.scale = scale
        self.branch0 = BasicConv2d(896, 128, 1, 1)
        self.branch1 = nn.Sequential(BasicConv2d(896, 128, 1, 1),
            BasicConv2d(128, 128, (1,7), 1, (0,3)), BasicConv2d(128, 128, (7,1), 1, (3,0)))
        self.conv2d = nn.Conv2d(256, 896, 1, 1)
        self.relu = nn.ReLU(inplace=False)
    def forward(self, x):
        o = self.conv2d(torch.cat((self.branch0(x), self.branch1(x)), 1))
        return self.relu(o * self.scale + x)

class Block8(nn.Module):
    def __init__(self, scale=1.0, noReLU=False):
        super().__init__()
        self.scale = scale; self.noReLU = noReLU
        self.branch0 = BasicConv2d(1792, 192, 1, 1)
        self.branch1 = nn.Sequential(BasicConv2d(1792, 192, 1, 1),
            BasicConv2d(192, 192, (1,3), 1, (0,1)), BasicConv2d(192, 192, (3,1), 1, (1,0)))
        self.conv2d = nn.Conv2d(384, 1792, 1, 1)
        if not noReLU: self.relu = nn.ReLU(inplace=False)
    def forward(self, x):
        o = self.conv2d(torch.cat((self.branch0(x), self.branch1(x)), 1))
        o = o * self.scale + x
        return o if self.noReLU else self.relu(o)

class Mixed_6a(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch0 = BasicConv2d(256, 384, 3, 2)
        self.branch1 = nn.Sequential(
            BasicConv2d(256, 192, 1, 1), BasicConv2d(192, 192, 3, 1, 1), BasicConv2d(192, 256, 3, 2))
        self.branch2 = nn.MaxPool2d(3, stride=2)
    def forward(self, x):
        return torch.cat((self.branch0(x), self.branch1(x), self.branch2(x)), 1)

class Mixed_7a(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch0 = nn.Sequential(
            BasicConv2d(896, 256, 1, 1), BasicConv2d(256, 384, 3, 2))
        self.branch1 = nn.Sequential(
            BasicConv2d(896, 256, 1, 1), BasicConv2d(256, 256, 3, 2))
        self.branch2 = nn.Sequential(
            BasicConv2d(896, 256, 1, 1), BasicConv2d(256, 256, 3, 1, 1), BasicConv2d(256, 256, 3, 2))
        self.branch3 = nn.MaxPool2d(3, stride=2)
    def forward(self, x):
        x3 = self.branch3(x)
        return torch.cat((self.branch0(x), self.branch1(x), self.branch2(x), x3), 1)

class InceptionResnetV1(nn.Module):
    def __init__(self, classify=False, num_classes=None, dropout_prob=0.6):
        super().__init__()
        self.classify = classify
        self.conv2d_1a = BasicConv2d(3, 32, 3, 2)
        self.conv2d_2a = BasicConv2d(32, 32, 3, 1)
        self.conv2d_2b = BasicConv2d(32, 64, 3, 1, 1)
        self.maxpool_3a = nn.MaxPool2d(3, 2)
        self.conv2d_3b = BasicConv2d(64, 80, 1, 1)
        self.conv2d_4a = BasicConv2d(80, 192, 3, 1)
        self.conv2d_4b = BasicConv2d(192, 256, 3, 2)
        self.repeat_1 = nn.Sequential(*[Block35(0.17) for _ in range(5)])
        self.mixed_6a = Mixed_6a()
        self.repeat_2 = nn.Sequential(*[Block17(0.10) for _ in range(10)])
        self.mixed_7a = Mixed_7a()
        self.repeat_3 = nn.Sequential(*[Block8(0.20) for _ in range(5)])
        self.block8 = Block8(noReLU=True)
        self.avgpool_1a = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout_prob)
        self.last_linear = nn.Linear(1792, 512, bias=False)
        self.last_bn = nn.BatchNorm1d(512, eps=0.001, momentum=0.1, affine=True)
        self.logits = nn.Linear(512, num_classes) if classify and num_classes else None
    def forward(self, x):
        x = self.conv2d_1a(x); x = self.conv2d_2a(x); x = self.conv2d_2b(x)
        x = self.maxpool_3a(x); x = self.conv2d_3b(x)
        x = self.conv2d_4a(x); x = self.conv2d_4b(x)
        x = self.repeat_1(x); x = self.mixed_6a(x); x = self.repeat_2(x)
        x = self.mixed_7a(x); x = self.repeat_3(x); x = self.block8(x)
        x = self.avgpool_1a(x); x = self.dropout(x)
        x = self.last_linear(x.view(x.shape[0], -1))
        x = self.last_bn(x)
        return self.logits(x) if (self.classify and self.logits) else F.normalize(x, p=2, dim=1)

print(f"Architecture: {sum(p.numel() for p in InceptionResnetV1().parameters()):,} params")

In [ ]:
# ═══ Load weights & verify ═════════════════════════════════════════
model = InceptionResnetV1(classify=False).to(DEVICE).eval()
sd = torch.load(WEIGHTS_PATH, map_location=DEVICE)

m_keys = set(model.state_dict().keys())
s_keys = set(sd.keys())
print(f"Keys: model={len(m_keys)}  weights={len(s_keys)}")

missing = m_keys - s_keys
extra = s_keys - m_keys
if missing: print(f"Missing: {len(missing)}")
if extra: print(f"Extra: {len(extra)}")

model.load_state_dict({k: v for k, v in sd.items() if k in m_keys}, strict=False)

with torch.no_grad():
    dummy = torch.randn(1, 3, 160, 160).to(DEVICE)
    emb = model(dummy)
    print(f"Output: {list(emb.shape)}  norm={emb.norm().item():.4f}")
print("OK.")

In [ ]:
# ═══ PyTorch → ONNX ════════════════════════════════════════════════
!pip install onnx onnxruntime -q 2>&1 | tail -1
import onnx, onnxruntime

ONNX_PATH = "facenet_mobile.onnx"
TFLITE_PATH = "facenet_mobile.tflite"

dummy_input = torch.randn(1, 3, 160, 160).to(DEVICE)
with torch.no_grad():
    pt_out = model(dummy_input).cpu()

torch.onnx.export(model, dummy_input, ONNX_PATH,
    input_names=["input"], output_names=["embedding"],
    dynamic_axes={"input":{0:"batch"},"embedding":{0:"batch"}},
    opset_version=13, dynamo=False)
print(f"ONNX: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB")

ort = onnxruntime.InferenceSession(ONNX_PATH)
ort_out = ort.run(None, {"input": dummy_input.cpu().numpy()})
diff = torch.abs(pt_out - torch.tensor(ort_out[0])).max().item()
print(f"ONNX vs PT diff: {diff:.6f} ({'OK' if diff < 1e-4 else 'WARN'})")

del model, dummy_input
torch.cuda.empty_cache()

In [ ]:
# ═══ ONNX → TFLite ═════════════════════════════════════════════════
print("ONNX → TFLite...")
!pip install onnx2tf ai-edge-torch -q 2>&1 | tail -1

success = False

# A: ai-edge-torch (Google's direct PyTorch → TFLite)
try:
    print("  A: ai-edge-torch...")
    import ai_edge_torch
    m = InceptionResnetV1(classify=False).eval()
    m.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu'))
    ai_edge_torch.convert(m, (torch.randn(1, 3, 160, 160),)).export(TFLITE_PATH)
    print(f"    TFLite: {os.path.getsize(TFLITE_PATH)/1e6:.1f} MB")
    success = True
except Exception as e:
    print(f"    {type(e).__name__}: {str(e)[:120]}")

# B: onnx2tf (with relaxed checks)
if not success:
    try:
        print("  B: onnx2tf...")
        import onnx2tf
        onnx2tf.convert(
            input_onnx_file_path=ONNX_PATH,
            output_folder_path='tflite_out',
            output_signaturedefs=True,
            copy_onnx_input_output_names_to_tflite=True,
            non_verbose=True,
            keep_nwc_or_nchw_or_ncdhw_input_names=['input'],
            keep_nwc_or_nchw_or_ncdhw_output_names=['embedding'],
            batch_norm_fusion=True,
        )
        import glob, shutil
        f = glob.glob('tflite_out/*.tflite')
        if f: shutil.copy(f[0], TFLITE_PATH)
        print(f"    TFLite: {os.path.getsize(TFLITE_PATH)/1e6:.1f} MB")
        success = True
    except Exception as e:
        print(f"    {type(e).__name__}: {str(e)[:120]}")

# C: Re-export ONNX with simpler opset, retry onnx2tf
if not success:
    try:
        print("  C: opset 11 ONNX → onnx2tf...")
        opset11_path = "facenet_opset11.onnx"
        m2 = InceptionResnetV1(classify=False).eval()
        m2.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
        torch.onnx.export(m2, torch.randn(1,3,160,160).to(DEVICE), opset11_path,
            input_names=["input"], output_names=["embedding"], opset_version=11, dynamo=False)
        import onnx2tf, glob, shutil
        onnx2tf.convert(input_onnx_file_path=opset11_path, output_folder_path='tflite_out2',
            output_signaturedefs=True, copy_onnx_input_output_names_to_tflite=True, non_verbose=True)
        f = glob.glob('tflite_out2/*.tflite')
        if f: shutil.copy(f[0], TFLITE_PATH)
        print(f"    TFLite: {os.path.getsize(TFLITE_PATH)/1e6:.1f} MB")
        success = True
    except Exception as e:
        print(f"    {type(e).__name__}: {str(e)[:120]}")

if success:
    print(f"\nSUCCESS: {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH)/1e6:.1f} MB)")
    print("Copy to: mobile/assets/models/facenet_mobile.tflite")
else:
    print(f"\nAll methods failed. ONNX file saved: {ONNX_PATH}")
    print(f"Size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB")
    print("\nDownload the ONNX file and convert locally:")
    print("  python -m pip install onnx2tf")
    print(f"  onnx2tf -i {ONNX_PATH} -osd")

In [ ]:
# ═══ Verify ════════════════════════════════════════════════════════
# Avoid importing tensorflow (breaks on numpy 2.x). Use tflite_runtime.
try:
    import tensorflow as tf
except ImportError:
    !pip install tflite-runtime -q 2>&1 | tail -1
    import tflite_runtime.interpreter as tf

interp = tf.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f"In: {inp['shape']}  Out: {out['shape']}")

test = np.random.randn(1, 160, 160, 3).astype(np.float32)
interp.set_tensor(inp['index'], test); interp.invoke()
r = interp.get_tensor(out['index'])
print(f"Norm: {np.linalg.norm(r[0]):.4f}")
print("\n✓ facenet_mobile.tflite ready — download from file sidebar")